# EfficientNet-B4 + BiLSTM Deepfake Detection (Kaggle-ready)

This notebook trains a **feature-based** EfficientNet-B4 + BiLSTM pipeline for deepfake detection.
It supports:
- mounting **your 140k real-vs-fake** crops dataset and **FaceForensics++ extracted frames** from Kaggle
- **precomputing EfficientNet-B4 features** per-frame and caching them per-video (.npz)
- training a **Bi-LSTM RNN head** on fixed-length sequences of features
- **MixUp** (feature-level) during training and **optional CutMix** (image-level) during precompute
- 70/15/15 video-level split (if train/valid/test CSVs present they will be used)
- metrics: frame/video AUC, precision, recall, accuracy and training/validation curves
- optional cross-dataset evaluation on Celeb-DF (if mounted)

Before running: add these datasets to the Notebook Data panel on Kaggle:
1. `140k-real-and-fake-faces` (your dataset) — appears as `/kaggle/input/140k-real-and-fake-faces`
2. `faceforencispp-extracted-frames` (Kaggle mirror) — appears as `/kaggle/input/faceforencispp-extracted-frames`
3. (optional) `celebd-faces` or similar for cross-dataset eval — appears as `/kaggle/input/<name>`

Edit the dataset folder names in the next cell if they differ.

Run cells in order. Feature precomputation is the slowest step; default CutMix is **off** (0.0).


In [2]:
# Cell: installs (uncomment if needed on Kaggle)
!pip install -q timm==0.9.2 albumentations[imgaug]==1.3.1 opencv-python-headless==4.7.0.72 facenet-pytorch==2.0.2 sklearn tqdm pandas
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())


ERROR: Could not find a version that satisfies the requirement facenet-pytorch==2.0.2 (from versions: 0.0.1, 0.1.0, 0.2.2, 0.2.3, 0.3.0, 0.3.1, 0.4.0, 0.5.0, 1.0.0, 1.0.1, 1.0.2, 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.2.0, 2.2.1, 2.2.2, 2.2.3, 2.2.4, 2.2.5, 2.2.6, 2.2.7, 2.2.8, 2.2.9, 2.3.0, 2.3.1, 2.4.1, 2.5.0, 2.5.1, 2.5.2, 2.5.3, 2.6.0)
ERROR: No matching distribution found for facenet-pytorch==2.0.2
torch 2.6.0+cu124 cuda: True


In [ ]:
import os, random, math, json, gc
from pathlib import Path
from glob import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt

# ----------------- EDIT IF NEEDED -----------------
USER_140K_NAME = '140k-real-and-fake-faces'  # your uploaded dataset folder name
FFPP_NAME = 'faceforencispp-extracted-frames'  # kaggle mirror folder name
CELEBDF_NAME = None  # set to your celebd frames dataset folder name if available, e.g. 'celebdf-frames'
# -------------------------------------------------
INPUT_ROOT = Path('/kaggle/input')
USER_140K_PATH = INPUT_ROOT / USER_140K_NAME
FFPP_PATH = INPUT_ROOT / FFPP_NAME
CELEBDF_PATH = INPUT_ROOT / CELEBDF_NAME if CELEBDF_NAME is not None else None

print('USER_140K_PATH exists?', USER_140K_PATH.exists())
print('FFPP_PATH exists?', FFPP_PATH.exists())
print('CELEBDF_PATH exists?', CELEBDF_PATH.exists() if CELEBDF_PATH is not None else False)

WORK_ROOT = Path('/kaggle/working/deepfake_project')
FEAT_ROOT = WORK_ROOT / 'features'
ARTIFACTS = WORK_ROOT / 'artifacts'
for p in [WORK_ROOT, FEAT_ROOT, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

RND_SEED = 42
random.seed(RND_SEED)
np.random.seed(RND_SEED)
torch.manual_seed(RND_SEED)


In [ ]:
# Cell: scan datasets for frames and build combined dataframe
def scan_dataset_for_frames(root_path, source_name):
    items = []
    for p in Path(root_path).rglob('*.jpg'):
        path_str = str(p)
        parts = [pp.name.lower() for pp in p.parents]
        label = None
        if any('real' in s or 'original' in s for s in parts):
            label = 0
        if any('fake' in s or 'manip' in s or 'deepf' in s for s in parts):
            label = 1
        fn = p.name.lower()
        if label is None:
            if 'fake' in fn or 'manip' in fn or 'deepf' in fn:
                label = 1
            elif 'real' in fn or 'orig' in fn:
                label = 0
        if label is None:
            continue
        stem_parts = p.stem.split('_')
        if len(stem_parts) >= 2 and 'frame' in stem_parts[-1]:
            video_id = '_'.join(stem_parts[:-1])
        else:
            if p.parent != Path(root_path):
                video_id = p.parent.name
            else:
                video_id = p.stem
        items.append((path_str, int(label), video_id, source_name))
    return pd.DataFrame(items, columns=['filepath','label','video_id','source'])

# scan user dataset
if not USER_140K_PATH.exists():
    raise RuntimeError(f"User dataset path {USER_140K_PATH} not found. Update USER_140K_NAME in the cell above.")
user_df = scan_dataset_for_frames(USER_140K_PATH, 'user140k')
print('Found user frames:', len(user_df))

ffpp_df = pd.DataFrame(columns=['filepath','label','video_id','source'])
if FFPP_PATH.exists():
    ffpp_df = scan_dataset_for_frames(FFPP_PATH, 'ffpp')
    print('Found FF++ frames:', len(ffpp_df))
else:
    print('No FF++ dataset found at', FFPP_PATH)

combined_df = pd.concat([user_df, ffpp_df], axis=0).reset_index(drop=True)
print('Combined frames total:', len(combined_df))

# If user provided train/valid/test CSVs in the user dataset root, prefer those splits
user_csv_dir = USER_140K_PATH
train_csv = user_csv_dir / 'train.csv'
valid_csv = user_csv_dir / 'valid.csv'
test_csv = user_csv_dir / 'test.csv'
use_provided_splits = False
if train_csv.exists() and valid_csv.exists() and test_csv.exists():
    try:
        train_df = pd.read_csv(train_csv)
        val_df = pd.read_csv(valid_csv)
        test_df = pd.read_csv(test_csv)
        # try to standardize columns if necessary
        for df_ in [train_df, val_df, test_df]:
            if 'filepath' not in df_.columns:
                # assume single column lists
                df_.columns = ['filepath']
                # try to infer labels from parent folder
                df_['label'] = df_['filepath'].apply(lambda p: 1 if 'fake' in str(p).lower() else 0)
        use_provided_splits = True
        print('Using provided train/valid/test CSVs from user dataset root')
    except Exception as e:
        print('Failed to load provided CSVs, will create splits instead:', e)

if not use_provided_splits:
    # create video-level 70/15/15 split using GroupShuffleSplit
    combined_df['video_uid'] = combined_df['source'] + '__' + combined_df['video_id']
    video_df = combined_df[['video_uid','label']].drop_duplicates().reset_index(drop=True)
    gss = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=RND_SEED)
    train_idx, rest_idx = next(gss.split(video_df['video_uid'], video_df['label'], groups=video_df['video_uid']))
    train_vids = video_df.loc[train_idx, 'video_uid'].tolist()
    rest_vids = video_df.loc[rest_idx, 'video_uid'].tolist()
    rest_df = video_df[video_df.video_uid.isin(rest_vids)].reset_index(drop=True)
    val_idx, test_idx = next(GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=RND_SEED).split(rest_df['video_uid'], rest_df['label'], groups=rest_df['video_uid']))
    val_vids = rest_df.loc[val_idx, 'video_uid'].tolist()
    test_vids = rest_df.loc[test_idx, 'video_uid'].tolist()
    train_df = combined_df[combined_df.video_uid.isin(train_vids)].reset_index(drop=True)
    val_df   = combined_df[combined_df.video_uid.isin(val_vids)].reset_index(drop=True)
    test_df  = combined_df[combined_df.video_uid.isin(test_vids)].reset_index(drop=True)

print('Train/Val/Test frames:', len(train_df), len(val_df), len(test_df))
train_df.to_csv(WORK_ROOT / 'train_frames.csv', index=False)
val_df.to_csv(WORK_ROOT / 'val_frames.csv', index=False)
test_df.to_csv(WORK_ROOT / 'test_frames.csv', index=False)


In [ ]:
# Cell: transforms and sequence dataset
IMG_SIZE = 320
SEQ_LEN = 16

train_transform = A.Compose([
    A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.6,1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.08, rotate_limit=10, p=0.5),
    A.ColorJitter(0.2,0.2,0.2,0.02, p=0.6),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])
class VideoFrameSequenceSimple(Dataset):
    def __init__(self, frames_csv, seq_len=SEQ_LEN, transform=None):
        self.df = pd.read_csv(frames_csv)
        self.seq_len = seq_len
        self.transform = transform
        self.vid2paths = {}
        for vid, g in self.df.groupby('video_uid'):
            self.vid2paths[vid] = {'paths': sorted(g.filepath.tolist()), 'label': int(g.label.iloc[0])}
        self.vids = list(self.vid2paths.keys())
    def __len__(self):
        return len(self.vids)
    def sample_paths(self, paths):
        n = len(paths)
        if n >= self.seq_len:
            idxs = np.linspace(0, n-1, num=self.seq_len, dtype=int).tolist()
        else:
            idxs = list(range(n)) + [n-1]*(self.seq_len - n)
        return [paths[i] for i in idxs]
    def __getitem__(self, idx):
        vid = self.vids[idx]
        info = self.vid2paths[vid]
        sel = self.sample_paths(info['paths'])
        imgs = []
        for p in sel:
            img = cv2.imread(p)
            if img is None:
                img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            if self.transform is not None:
                out = self.transform(image=img)
                img_t = out['image']
            else:
                img_t = ToTensorV2()(image=img)['image']
            imgs.append(img_t)
        seq = torch.stack(imgs, dim=0)
        label = torch.tensor(float(info['label']))
        return seq, label, vid


In [ ]:
# Cell: EfficientNet feature extractor, CutMix helper, and precompute features
MODEL_NAME = 'tf_efficientnet_b4_ns'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device for extraction:', DEVICE)
class EfficientFeatureExtractor(nn.Module):
    def __init__(self, model_name=MODEL_NAME, pretrained=True):
        super().__init__()
        try:
            self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
            # timm returns features when num_classes=0
        except Exception:
            # fallback: create model and remove classifier
            full = timm.create_model(model_name, pretrained=pretrained)
            if hasattr(full, 'fc'):
                # resnet style
                full.fc = nn.Identity()
            elif hasattr(full, 'classifier'):
                full.classifier = nn.Identity()
            elif hasattr(full, 'global_pool'):
                # many timm nets have global_pool; set to identity
                pass
            self.model = full

    def forward(self, x):
        return self.model(x)

# CutMix helper (image-level). We'll use it optionally during feature precomputation.
def cutmix_image(img1, img2, alpha=1.0):
    # img1, img2: HWC uint8 or float [0,1]
    H, W = img1.shape[:2]
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    # sample bounding box
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    out = img1.copy()
    out[y1:y2, x1:x2, :] = img2[y1:y2, x1:x2, :]
    # recompute lam as proportion
    lam_eff = 1.0 - ((x2 - x1) * (y2 - y1) / float(W * H))
    return out, lam_eff

def precompute_and_save_all(feature_model, video_map, feat_root, batch_size=64, cutmix_prob=0.0, cutmix_alpha=1.0):
    feature_model.eval()
    for vid, paths in tqdm(video_map.items(), desc='Precompute'):
        outp = Path(feat_root) / f"{vid}.npz"
        if outp.exists():
            continue
        feats_list = []
        imgs_acc = []
        for p in paths:
            img = cv2.imread(p)
            if img is None:
                img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # optional CutMix: pick another random image from dataset
            if cutmix_prob > 0 and np.random.rand() < cutmix_prob:
                # pick random donor path from video_map
                donor_vid = np.random.choice(list(video_map.keys()))
                donor_paths = video_map[donor_vid]
                donor_p = np.random.choice(donor_paths)
                donor_img = cv2.imread(donor_p)
                if donor_img is None:
                    donor_img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
                donor_img = cv2.cvtColor(donor_img, cv2.COLOR_BGR2RGB)
                img, lam_eff = cutmix_image(img, donor_img, alpha=cutmix_alpha)
            proc = val_transform(image=img)['image']
            imgs_acc.append(proc.unsqueeze(0))
            if len(imgs_acc) == batch_size:
                batch = torch.cat(imgs_acc, dim=0).to(DEVICE)
                with torch.no_grad():
                    f = feature_model(batch).cpu().numpy()
                feats_list.append(f)
                imgs_acc = []
        if len(imgs_acc) > 0:
            batch = torch.cat(imgs_acc, dim=0).to(DEVICE)
            with torch.no_grad():
                f = feature_model(batch).cpu().numpy()
            feats_list.append(f)
        if len(feats_list) > 0:
            feats_all = np.concatenate(feats_list, axis=0)
        else:
            feats_all = np.zeros((0, feature_model.model.num_features if hasattr(feature_model.model, 'num_features') else 1536), dtype=np.float32)
        np.savez_compressed(outp, feats=feats_all, paths=np.array(paths, dtype=object))
    print('Precompute complete')


In [ ]:
# Cell: prepare video_map and run feature precomputation (this is the slowest step)
feature_model = EfficientFeatureExtractor().to(DEVICE)
feature_model.eval()

# Build video->paths map from train/val/test
video_map = {}
for df_ in [train_df, val_df, test_df]:
    for vid, g in df_.groupby('video_uid'):
        if vid in video_map:
            continue
        video_map[vid] = sorted(g.filepath.tolist())

print('Videos to precompute:', len(video_map))
# Set cutmix_prob to 0.0 by default; change to 0.2 to enable image-level CutMix during precompute
CUTMIX_PROB = 0.0
CUTMIX_ALPHA = 1.0
precompute_and_save_all(feature_model, video_map, FEAT_ROOT, batch_size=64, cutmix_prob=CUTMIX_PROB, cutmix_alpha=CUTMIX_ALPHA)


In [ ]:
# Cell: FeatureDataset and DataLoaders
class VideoFeatureDataset(Dataset):
    def __init__(self, frames_csv, features_root, seq_len=SEQ_LEN):
        self.df = pd.read_csv(frames_csv)
        self.features_root = Path(features_root)
        self.seq_len = seq_len
        self.vids = []
        self.vid2info = {}
        for vid, g in self.df.groupby('video_uid'):
            fpath = self.features_root / f"{vid}.npz"
            if not fpath.exists():
                continue
            feats = np.load(fpath, allow_pickle=True)['feats']
            label = int(g.label.iloc[0])
            self.vids.append(vid)
            self.vid2info[vid] = {'feats': feats, 'label': label}
    def __len__(self):
        return len(self.vids)
    def sample_seq(self, feats):
        n = feats.shape[0]
        if n >= self.seq_len:
            idxs = np.linspace(0, n-1, num=self.seq_len, dtype=int).tolist()
        else:
            idxs = list(range(n)) + [n-1]*(self.seq_len - n)
        return feats[idxs]
    def __getitem__(self, idx):
        vid = self.vids[idx]
        info = self.vid2info[vid]
        seq = self.sample_seq(info['feats'])
        seq = torch.from_numpy(seq).float()
        label = torch.tensor(float(info['label']))
        return seq, label, vid

train_feat_ds = VideoFeatureDataset(str(WORK_ROOT / 'train_frames.csv'), FEAT_ROOT, seq_len=SEQ_LEN)
val_feat_ds = VideoFeatureDataset(str(WORK_ROOT / 'val_frames.csv'), FEAT_ROOT, seq_len=SEQ_LEN)
test_feat_ds = VideoFeatureDataset(str(WORK_ROOT / 'test_frames.csv'), FEAT_ROOT, seq_len=SEQ_LEN)
print('Train vids:', len(train_feat_ds), 'Val vids:', len(val_feat_ds), 'Test vids:', len(test_feat_ds))
BATCH_SIZE = 32
train_loader = DataLoader(train_feat_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_feat_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_feat_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
# Cell: MixUp helper and BiLSTM head
def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index,:]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, preds, y_a, y_b, lam):
    return lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)

sample_seq, _, _ = next(iter(train_loader))
feat_dim = sample_seq.shape[-1]
print('Feature dim:', feat_dim)

class BiLSTMHead(nn.Module):
    def __init__(self, feat_dim, hidden_size=512, num_layers=1, bidirectional=True, dropout=0.3):
        super().__init__()
        self.rnn = nn.LSTM(input_size=feat_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, bidirectional=bidirectional, dropout=dropout if num_layers>1 else 0.0)
        rnn_out_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.Linear(rnn_out_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,1))
    def forward(self, seq):
        rnn_out, _ = self.rnn(seq)
        final = rnn_out[:, -1, :]
        logits = self.head(final).squeeze(1)
        return logits

model = BiLSTMHead(feat_dim=feat_dim).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)
scaler = torch.cuda.amp.GradScaler()


In [ ]:
# Cell: training and evaluation (MixUp applied on feature sequences)
from collections import defaultdict
def train_epoch(model, loader, optimizer, criterion, device, mixup_alpha=0.4):
    model.train()
    losses = []
    preds, trues = [], []
    for seqs, labels, vids in tqdm(loader, desc='Train'):
        seqs = seqs.to(device)  # (B, T, feat_dim)
        labels = labels.to(device)
        optimizer.zero_grad()
        if mixup_alpha > 0:
            mixed_x, y_a, y_b, lam = mixup_data(seqs, labels.unsqueeze(1), alpha=mixup_alpha)
            mixed_x = mixed_x.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(mixed_x)
                # y_a, y_b are shape (B,1) -> squeeze
                y_a = y_a.to(device).squeeze(1)
                y_b = y_b.to(device).squeeze(1)
                loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
        else:
            with torch.cuda.amp.autocast():
                outputs = model(seqs)
                loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
        probs = torch.sigmoid(outputs).detach().cpu().numpy().ravel().tolist()
        # if mixup was used, labels used for metrics are y_a*lam + y_b*(1-lam) => approximate by outputs
        trues += labels.detach().cpu().numpy().ravel().tolist()
        preds += probs
    avg_loss = float(np.mean(losses))
    try:
        auc = roc_auc_score(trues, preds)
    except:
        auc = 0.5
    return avg_loss, auc

def evaluate_video_level(model, loader, device):
    model.eval()
    video_scores = defaultdict(list)
    video_labels = {}
    with torch.no_grad():
        for seqs, labels, vids in tqdm(loader, desc='Eval'):
            seqs = seqs.to(device)
            outputs = model(seqs)
            probs = torch.sigmoid(outputs).cpu().numpy().ravel().tolist()
            for p, y, vid in zip(probs, labels.numpy().ravel().tolist(), vids):
                video_scores[vid].append(p)
                video_labels[vid] = y
    vids_list = sorted(video_scores.keys())
    vid_preds = [np.mean(video_scores[v]) for v in vids_list]
    vid_trues = [video_labels[v] for v in vids_list]
    try:
        video_auc = roc_auc_score(vid_trues, vid_preds)
    except:
        video_auc = 0.5
    # frame-level metrics (approx)
    frame_preds = []
    frame_trues = []
    for v in vids_list:
        frame_preds += video_scores[v]
        frame_trues += [video_labels[v]]*len(video_scores[v])
    try:
        frame_auc = roc_auc_score(frame_trues, frame_preds)
    except:
        frame_auc = 0.5
    # precision/recall/acc at threshold 0.5 on video preds
    y_pred_bin = [1 if p>0.5 else 0 for p in vid_preds]
    try:
        prec = precision_score(vid_trues, y_pred_bin)
        rec = recall_score(vid_trues, y_pred_bin)
        acc = accuracy_score(vid_trues, y_pred_bin)
    except:
        prec, rec, acc = 0.0, 0.0, 0.0
    return {'video_auc': video_auc, 'frame_auc': frame_auc, 'precision': prec, 'recall': rec, 'accuracy': acc}


In [ ]:
# Cell: Run training
EPOCHS = 8
history = {'train_loss':[], 'train_auc':[], 'val_video_auc':[], 'val_frame_auc':[], 'val_prec':[], 'val_rec':[], 'val_acc':[]}
best_val = 0.0
ckpt = ARTIFACTS / 'best_model.pth'
for epoch in range(1, EPOCHS+1):
    print('\nEpoch', epoch)
    tr_loss, tr_auc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, mixup_alpha=0.4)
    scheduler.step()
    val_metrics = evaluate_video_level(model, val_loader, DEVICE)
    print(f"Train loss: {tr_loss:.4f} Train AUC: {tr_auc:.4f} | Val video AUC: {val_metrics['video_auc']:.4f} frame AUC: {val_metrics['frame_auc']:.4f} prec: {val_metrics['precision']:.4f} rec: {val_metrics['recall']:.4f} acc: {val_metrics['accuracy']:.4f}")
    history['train_loss'].append(tr_loss)
    history['train_auc'].append(tr_auc)
    history['val_video_auc'].append(val_metrics['video_auc'])
    history['val_frame_auc'].append(val_metrics['frame_auc'])
    history['val_prec'].append(val_metrics['precision'])
    history['val_rec'].append(val_metrics['recall'])
    history['val_acc'].append(val_metrics['accuracy'])
    if val_metrics['video_auc'] > best_val:
        best_val = val_metrics['video_auc']
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'optimizer': optimizer.state_dict(), 'history': history}, ckpt)
        print('Saved checkpoint', ckpt)
print('Best val video AUC:', best_val)


In [ ]:
# Cell: Test evaluation + optional Celeb-DF cross-eval
ck = torch.load(ckpt, map_location='cpu')
model.load_state_dict(ck['model_state'])
model.to(DEVICE).eval()
test_metrics = evaluate_video_level(model, test_loader, DEVICE)
print('Test set metrics:', test_metrics)

if CELEBDF_PATH is not None and CELEBDF_PATH.exists():
    print('Preparing Celeb-DF frames for cross-eval...')
    celebd_df = scan_dataset_for_frames(CELEBDF_PATH, 'celebdf')
    if len(celebd_df) > 0:
        celebd_df['video_uid'] = celebd_df['source'] + '__' + celebd_df['video_id']
        celebd_df.to_csv(WORK_ROOT / 'celebd_frames.csv', index=False)
        celebd_vmap = {vid: sorted(g.filepath.tolist()) for vid,g in celebd_df.groupby('video_uid')}
        precompute_and_save_all(feature_model, celebd_vmap, FEAT_ROOT, batch_size=64, cutmix_prob=0.0)
        celebd_feat_ds = VideoFeatureDataset(str(WORK_ROOT / 'celebd_frames.csv'), FEAT_ROOT, seq_len=SEQ_LEN)
        celebd_loader = DataLoader(celebd_feat_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        celebd_metrics = evaluate_video_level(model, celebd_loader, DEVICE)
        print('Celeb-DF cross-dataset metrics:', celebd_metrics)
    else:
        print('No frames in CELEBDF_PATH')
else:
    print('CELEBDF_PATH not provided or not found; skipping cross-dataset eval')


In [ ]:
# Cell: plots and save artifacts
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='train_loss')
plt.title('Train Loss')
plt.xlabel('epoch')
plt.legend()
plt.subplot(1,2,2)
plt.plot(history['val_video_auc'], label='val_video_auc')
plt.plot(history['train_auc'], label='train_auc')
plt.title('AUC')
plt.xlabel('epoch')
plt.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS / 'training_curves.png')
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history['val_prec'], label='val_precision')
plt.plot(history['val_rec'], label='val_recall')
plt.plot(history['val_acc'], label='val_accuracy')
plt.title('Val precision/recall/acc')
plt.xlabel('epoch')
plt.legend()
plt.savefig(ARTIFACTS / 'pr_acc_curves.png')
plt.show()

with open(ARTIFACTS / 'history.json', 'w') as f:
    json.dump(history, f)
with open(ARTIFACTS / 'model_card.txt', 'w') as f:
    f.write(f"Model card - best_val: {best_val}\nSEQ_LEN: {SEQ_LEN}\nIMG_SIZE: {IMG_SIZE}\n")
print('Saved artifacts to', ARTIFACTS)
